# Six-class baseline comparison
Uses the saved fit and validation row indices from preprocessing. Every learned transformer is inside a pipeline fitted on the fit partition. The official test data is reserved for final evaluation.

In [ ]:
from pathlib import Path
import json, os, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
RESULTS = ROOT / 'results'
PLOTS = RESULTS / 'plots'
SEED = 42
CLASSES = ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers']
assert not (RESULTS / 'model_comparison.csv').exists() and not (RESULTS / 'experiment_log.csv').exists(), 'Existing results require review before rerun.'
PLOTS.mkdir(parents=True, exist_ok=True)
frame = pd.read_csv(DATA / 'selected_training.csv')
fit_idx = pd.read_csv(DATA / 'fit_indices.csv')['row_index'].to_numpy()
val_idx = pd.read_csv(DATA / 'validation_indices.csv')['row_index'].to_numpy()
assert len(set(fit_idx) & set(val_idx)) == 0
assert set(fit_idx) | set(val_idx) == set(range(len(frame)))
predictors = [c for c in frame if c not in {'id', 'attack_cat', 'label', '_fingerprint'}]
assert {'id', 'attack_cat', 'label', '_fingerprint'}.isdisjoint(predictors)
X_fit, X_val = frame.iloc[fit_idx][predictors], frame.iloc[val_idx][predictors]
y_fit, y_val = frame.iloc[fit_idx]['attack_cat'], frame.iloc[val_idx]['attack_cat']
assert set(y_fit) == set(y_val) == set(CLASSES)
numeric = X_fit.select_dtypes(include='number').columns.tolist()
categorical = X_fit.select_dtypes(exclude='number').columns.tolist()
print('Fit', X_fit.shape, 'Validation', X_val.shape, 'Numeric', len(numeric), 'Categorical', len(categorical))

## Baselines
The tree models use sparse one-hot features. HistGradientBoosting receives a dense transformation, capped here by the observed small categorical vocabulary. Each model saves its result immediately after evaluation.

In [ ]:
def make_preprocessor(dense=False):
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=not dense, dtype=np.float32))]), categorical),
    ], sparse_threshold=0 if dense else 1)

models = [
    ('Dummy', DummyClassifier(strategy='most_frequent'), False),
    ('Decision Tree', DecisionTreeClassifier(random_state=SEED), False),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=2), False),
    ('Extra Trees', ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=2), False),
    ('HistGradientBoosting', HistGradientBoostingClassifier(random_state=SEED), True),
]
summary_rows, log_rows = [], []
for name, estimator, dense in models:
    if name == 'Dummy':
        pipeline = Pipeline([('classifier', estimator)])
    else:
        pipeline = Pipeline([('preprocessor', make_preprocessor(dense)), ('classifier', estimator)])
    start = time.perf_counter()
    pipeline.fit(X_fit, y_fit)
    fit_seconds = time.perf_counter() - start
    prediction = pipeline.predict(X_val)
    report = classification_report(y_val, prediction, labels=CLASSES, output_dict=True, zero_division=0)
    matrix = confusion_matrix(y_val, prediction, labels=CLASSES)
    attack_true = y_val.to_numpy() != 'Normal'
    attack_pred = prediction != 'Normal'
    attack_recall = float(attack_pred[attack_true].mean())
    normal_false_positive_rate = float(attack_pred[~attack_true].mean())
    row = {'model': name, 'configuration': json.dumps(estimator.get_params(), sort_keys=True, default=str), 'fit_seconds': round(fit_seconds, 3), 'accuracy': accuracy_score(y_val, prediction), 'macro_f1': f1_score(y_val, prediction, average='macro'), 'weighted_f1': f1_score(y_val, prediction, average='weighted'), 'attack_recall': attack_recall, 'normal_false_positive_rate': normal_false_positive_rate}
    for cls in CLASSES:
        for metric in ('precision', 'recall', 'f1-score'):
            row[f'{cls}_{metric}'] = report[cls][metric]
    row['confusion_matrix'] = json.dumps(matrix.tolist())
    summary_rows.append(row)
    for cls in CLASSES:
        log_rows.append({'model': name, 'configuration': row['configuration'], 'class': cls, 'support': int(report[cls]['support']), 'precision': report[cls]['precision'], 'recall': report[cls]['recall'], 'f1': report[cls]['f1-score'], 'fit_seconds': row['fit_seconds']})
    pd.DataFrame(summary_rows).to_csv(RESULTS / 'model_comparison.csv', index=False)
    pd.DataFrame(log_rows).to_csv(RESULTS / 'experiment_log.csv', index=False)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(matrix, cmap='Blues')
    ax.set(xticks=range(6), yticks=range(6), xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted', ylabel='Actual', title=name)
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right')
    for i in range(6):
        for j in range(6):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=8)
    fig.tight_layout()
    fig.savefig(PLOTS / f'{name.lower().replace(' ', '_')}_confusion.png', dpi=130)
    plt.close(fig)
    print(name, 'macro-F1', round(row['macro_f1'], 4), 'attack recall', round(attack_recall, 4), 'normal FPR', round(normal_false_positive_rate, 4), 'seconds', round(fit_seconds, 1), flush=True)

## Comparison and preliminary decisions
Recommendations balance macro-F1, attack detection, and normal false alarms. They are preliminary baseline decisions; no hyperparameter search is performed here.

In [ ]:
comparison = pd.DataFrame(summary_rows)
best = comparison.loc[comparison['macro_f1'].idxmax(), 'model']
comparison['recommendation'] = comparison['model'].map(lambda name: 'DISCARD' if name == 'Dummy' else ('KEEP' if name == best else 'TUNE'))
comparison['reason'] = comparison.apply(lambda row: 'Majority-class reference; misses attacks' if row.model == 'Dummy' else ('Highest baseline macro-F1; inspect false alarms' if row.recommendation == 'KEEP' else 'Compare tuning potential against attack recall and normal false alarms'), axis=1)
comparison.to_csv(RESULTS / 'model_comparison.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 4))
comparison.set_index('model')[['macro_f1', 'attack_recall', 'normal_false_positive_rate']].plot.bar(ax=ax, rot=20)
ax.set_ylim(0, 1); ax.set_ylabel('Score / rate'); fig.tight_layout()
fig.savefig(PLOTS / 'baseline_comparison.png', dpi=140); plt.close(fig)
display(comparison[['model', 'accuracy', 'macro_f1', 'weighted_f1', 'attack_recall', 'normal_false_positive_rate', 'fit_seconds', 'recommendation', 'reason']])